# Rare-event identification (REI)

Find spatially coherent rare regions in a CPFE field (for example high
Nye-tensor or von-Mises stress) with graph spatial clustering, a hierarchical
merge, and a rare-cluster selection, then export the result to VTK. Pure Python
(networkit Leiden, scipy hierarchical clustering, PyVista/VTK) — so it runs from
`pip install graintrace`.

<div class="admonition note">
<p class="admonition-title">Not auto-run in these docs</p>
<p>The full pipeline (graph over-segmentation + Leiden + hierarchical merge +
VTK export) is runtime-heavy, so it is <strong>not executed</strong> when the
docs build. It runs locally and in Colab (the badge above). For quick,
executed examples see the 2D and 3D tutorials.</p>
</div>

See the algorithm page :doc:`/algorithms/rare-event-identification` and the API
:class:`~graintrace.IdentifyRareClusters`.

## Inputs

A point-cloud CSV with an `id` column, `x, y, z` coordinates, and field columns.
REI is not grid-locked: a regular grid uses the fast grid graph, and arbitrary
points fall back to the kNN graph. Two CPFE sources share the same schema:

- `mesh_out/out_element_centroid_*.csv` — crisp per-element fields on the true
  mesh (`mesh_csv="sync"` default), full fidelity, one row per element (kNN path).
  Preferred.
- `grid_out/out_element_centroid_*.csv` — a regular grid, from
  `grid_transfer="per_step"` or an offline `GridResampler` (smoothed; see
  [post-processing](post-processing.ipynb)).

The example script regenerates `mwe_data/synthetic_vms.csv` when
`generate_synthetic=True`.

## Build the metric and stage objects

Stage 1 (`GraphSpatialCluster`) over-segments the field into many small clusters;
stage 2 (`ClusterAnalysisIndicator`) merges them hierarchically on per-cluster
means. The reduced metric (`spec_reduced`) is the stage-1 feature aggregated to
cluster means.

In [ ]:
import pandas as pd
from graintrace.rare_cluster_indicator import IdentifyRareClusters
from graintrace.similarity_metric_library import SimilarityMetricLibrary
from graintrace.user_data_class import SimilarityMetric, WeightConfig, RareCriteria
from graintrace import rare_criteria_selection_library as rcs

filename = "mwe_data/synthetic_vms.csv"

metric_lib = SimilarityMetricLibrary()
spec = metric_lib.von_mises_stress()
spec_reduced = SimilarityMetric(
    name=spec.name + "_mean",
    feature_cols=[f"{c}_mean" for c in spec.feature_cols],
    func=spec.func,
)

weight_cfg = WeightConfig(
    mode="rbf", power=2.0, sigma=None,
    sigma_auto={"sample_size": 20_000, "random_state": 42, "quantile": 0.5},
)

rare_criteria = RareCriteria(
    selector=lambda df: rcs.select_highest_scalar(
        df, k=3, required_cols="von_mises_stress_mean_mean", min_size=1
    )
)

irc = IdentifyRareClusters(
    input_csv_path=filename, id_col="id", coord_cols=("x", "y", "z")
)
gsc, indicator = irc.make_stage_objects(graph_cluster_out="rei_reduced.csv")

## Run the two-stage clustering

`networkit_kwargs["gamma"]` sets the Leiden resolution (higher gives more, smaller
clusters for stage-1 over-segmentation); `threshold` cuts the stage-2 dendrogram.

In [ ]:
bundle = irc.run_clustering(
    gsc=gsc,
    indicator=indicator,
    reduced_csv_path="rei_reduced.csv",
    gsc_run_kwargs=dict(
        spec=spec, graph_mode="grid", manhattan_radius=2, grid_tol=1e-6,
        n_jobs=1, weight_chunk_size=500_000, segmenter="leiden", seed=42,
        weight_cfg=weight_cfg, reduce_edges_topweights_k=20,
        networkit_kwargs={"gamma": 10.0}, resume_from_checkpoint=False,
    ),
    indicator_run_kwargs=dict(
        method_type="scipy_hierarchical", spec=spec_reduced,
        threshold=0.0005, method="average", criterion="distance",
    ),
)

## Select and export the rare clusters

`RareCriteria` picks the rare clusters from the per-cluster statistics (here the
three with the highest mean von-Mises); the result is written to VTK as labeled
blocks.

In [ ]:
out = irc.run_get_rare_cluster(
    bundle=bundle,
    criteria=rare_criteria,
    output_vtk_path="rare_clusters.vtk",
    export_control="auto",
    background_block_id=1,
    first_rare_block_id=2,
    also_write_final_label=True,
)
print("VTK exported:", out["output_vtk_path"])

## Notes

- Three checkpoint levels are available (bundle pickle, reduced CSV + labels,
  graph edges) for restarting long runs; see the REI restart pattern in
  [configuration](../configuration.rst).
- The clustering pipeline uses no multiprocessing, so no
  `if __name__ == "__main__":` guard is required.

**See also**

- Algorithm: :doc:`/algorithms/rare-event-identification`
- Quick executed examples: :doc:`rei-example-2d`, :doc:`rei-example-3d`
- Compare two REI outputs: :doc:`rei-comparison`
- Full script: [examples/demonstrate_rei_pipeline.py](https://github.com/applied-material-modeling/graintrace/blob/main/examples/demonstrate_rei_pipeline.py)